# 🎬 Render Penuh 3D Cinema Full HD 9:16 — *3 Kota Paling Unik di Dunia* (Google Colab Pro — NVIDIA L4 OptiX)

Notebook ini dirancang khusus untuk **Google Colab Pro (GPU NVIDIA L4 24GB VRAM, RT Cores Ada Lovelace — OptiX)** untuk merender keseluruhan animasi **76,1 detik (2.283 frame @ 30fps)** pada resolusi penuh **Full HD Vertikal `1080×1920` (9:16)** dengan peningkatan kualitas maksimal:
- **OptiX Hardware Raytracing (L4)** + **OpenImageDenoise (`RGB + Albedo + Normal`)** menjaga detail tekstur batu, stucco, & kayu tetap tajam.
- **8-Sample Ambient Occlusion** & **8-Bounce Cycles Global Illumination** (`diffuse=4`, `glossy=4`, `transmission=6`) untuk bayangan kontak & pantulan cahaya natural.
- **Physical Clearcoat & Specular Highlight Boost** pada logam kuningan/tembaga, cat truk 4x4, lantai linoleum, marmer kafe, kristal Opal pelangi, salju Arktik, & jalan batu Setenil.
- **Blender Compositor Cinema Pass** (*Fog Glow* highlight bloom untuk jendela hangat Begich Towers, neon terowongan, sorot Opal, & lentera Golden Hour Setenil + filmic *Lift/Gamma/Gain* grading).
- **Sprite Karakter 2.5D Ultra-Tajam (`scale=3.0` → `720×1560`)** dirender paralel di 12 vCPU.
- **Auto-Resume Checkpoint ke Google Drive**: jika sesi terputus, jalankan ulang Cell 6 saja — frame yang sudah selesai otomatis dilewati!

> **Penting sebelum mulai:** Pastikan di menu atas Colab pilih **Runtime → Change runtime type → Hardware accelerator: `L4 GPU`**!

## 1. Verifikasi GPU NVIDIA L4 & Mount Google Drive (Auto-Checkpoint)

In [ ]:
import os, subprocess
from google.colab import drive

gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
print('=== GPU TERDETEKSI ===')
print(gpu_info)
if 'L4' not in gpu_info:
    print('PERINGATAN: GPU bukan L4! Disarankan pindah ke Runtime -> Change runtime type -> L4 GPU untuk kecepatan & kualitas optimal.')

print('\n=== MOUNT GOOGLE DRIVE ===')
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/3_Kota_Unik_L4_Render'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Folder checkpoint Google Drive siap: {DRIVE_ROOT}')

## 2. Clone Repository, Install Blender 4.2.3 LTS & Cek OptiX L4
*(Membersihkan `PYTHONPATH` Python 3.13 Colab agar tidak bentrok dengan Python 3.11 internal Blender 4.2!)*

In [ ]:
%%bash
set -e

# 1. Clone atau update repository proyek
REPO_DIR="/content/animasi-niat"
if [ ! -d "$REPO_DIR/.git" ]; then
    git clone https://github.com/Genesys-ux/animasi-niat.git "$REPO_DIR"
else
    cd "$REPO_DIR" && git pull origin main
fi

# 2. Install dependensi Python untuk Character Engine 2.5D & assembly video
pip install -q pillow numpy resvg-py

# 3. Install runtime library Linux lengkap untuk Blender 4.2 headless + GPU
apt-get update -qq && apt-get install -y -qq \
    libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libegl1 libglu1-mesa \
    libxkbcommon0 libsm6 libice6 libx11-6 libxext6 libgomp1 ffmpeg > /dev/null

# 4. Download Blender 4.2.3 LTS jika belum terpasang dengan baik
if ! env -u PYTHONPATH -u PYTHONHOME /content/blender-4.2/blender --version >/dev/null 2>&1; then
    echo "Mengunduh Blender 4.2.3 LTS..."
    rm -rf /content/blender-4.2 /tmp/blender.tar.xz
    curl -fsSL https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -o /tmp/blender.tar.xz
    mkdir -p /content/blender-4.2
    tar -xf /tmp/blender.tar.xz -C /content/blender-4.2 --strip-components=1
    rm -f /tmp/blender.tar.xz
fi

# 5. Verifikasi Blender Python 3.11 + Deteksi GPU OptiX L4!
env -u PYTHONPATH -u PYTHONHOME -u PYTHONSTARTUP -u PYTHONUSERBASE \
    /content/blender-4.2/blender --background --python-exit-code 1 --python-expr "
import bpy
print('Blender Python OK:', bpy.app.version_string)
p = bpy.context.preferences.addons['cycles'].preferences
for bk in ('OPTIX', 'CUDA'):
    try:
        p.compute_device_type = bk
        p.get_devices()
        devs = [d.name for d in p.devices if d.type == bk]
        print(f'Backend {bk}:', devs)
    except Exception as e:
        print(f'Backend {bk} error:', e)
"
echo "Setup Blender 4.2.3 LTS & GPU L4 SELESAI!"

## 3. Konfigurasi Kualitas Render Cinema L4
Sesuaikan parameter di bawah ini bila diperlukan (pengaturan default sudah optimal untuk **Full HD 1080×1920 Cinema di GPU L4** — estimasi ~65–95 menit total dengan auto-resume).

In [ ]:
#@title Pengaturan Render Cinema NVIDIA L4 { run: "auto" }
RESOLUTION = "1080x1920"  #@param ["1080x1920", "720x1280", "540x960"]
SAMPLES = 64              #@param [32, 48, 64, 96, 128] {type:"raw"}
QUALITY_MODE = "cinema"   #@param ["cinema", "draft"]
DEVICE = "OPTIX"          #@param ["OPTIX", "CUDA", "AUTO"]
SPRITE_SCALE = "3.0"      #@param ["2.0", "2.5", "3.0"]
START_FRAME = 0           #@param {type:"integer"}
END_FRAME = 2282          #@param {type:"integer"}
SAVE_TO_DRIVE = True      #@param {type:"boolean"}
CRF_VIDEO = 16            #@param {type:"integer"}

import os, subprocess
from collections import deque

SPRITE_SCALE_F = float(SPRITE_SCALE)
REPO_DIR = "/content/animasi-niat"
BLENDER_BIN = "/content/blender-4.2/blender"
DRIVE_ROOT = "/content/drive/MyDrive/3_Kota_Unik_L4_Render"

def blender_env(cwd=REPO_DIR):
    """Hapus PYTHONPATH/PYTHONHOME Python 3.13 Colab & sinkronkan PWD agar path Blender 4.2 bersih 100%."""
    drop = {'PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'PYTHONUSERBASE'}
    env = {k: v for k, v in os.environ.items() if k not in drop}
    env['PWD'] = cwd
    return env

def run_logged(cmd, cwd=REPO_DIR, use_blender_env=False, filter_keywords=None):
    """Jalankan perintah dengan output stdout+stderr real-time & dump penuh jika error!"""
    env = blender_env(cwd) if use_blender_env else {**os.environ, 'PWD': cwd}
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = deque(maxlen=50)
    for line in proc.stdout:
        tail.append(line)
        if filter_keywords is None or any(k in line for k in filter_keywords):
            print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        print('\n================ ERROR LOG (50 BARIS TERAKHIR) ================')
        print(''.join(tail))
        raise RuntimeError(f'Perintah gagal (exit code {ret}): {" ".join(cmd)}')

if SAVE_TO_DRIVE:
    FRAMES_DIR = f"{DRIVE_ROOT}/frames_{RESOLUTION}_{QUALITY_MODE}_s{SAMPLES}"
    OUT_DIR = f"{DRIVE_ROOT}/output"
else:
    FRAMES_DIR = f"{REPO_DIR}/production/rendered_frames_{RESOLUTION}"
    OUT_DIR = f"{REPO_DIR}/production/output"

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_MP4 = f"{OUT_DIR}/3_Kota_Paling_Unik_FULL_HD_9x16_L4_{QUALITY_MODE}_s{SAMPLES}.mp4"
CPU_WORKERS = min(12, os.cpu_count() or 8)

print(f"Resolusi        : {RESOLUTION} (Vertikal 9:16)")
print(f"Samples / Device: {SAMPLES} samples @ Cycles {DEVICE} ({QUALITY_MODE.upper()})")
print(f"Sprite Scale    : {SPRITE_SCALE_F}x ({int(240*SPRITE_SCALE_F)}x{int(520*SPRITE_SCALE_F)} px) @ {CPU_WORKERS} workers")
print(f"Rentang Frame   : {START_FRAME} .. {END_FRAME} ({END_FRAME - START_FRAME + 1} frames = {(END_FRAME - START_FRAME + 1)/30:.2f}s)")
print(f"Folder Frames   : {FRAMES_DIR}")
print(f"Output MP4      : {OUT_MP4}")

## 4. Generate Tekstur Papan Nama, Build Scene 3D Master & Render Sprite Karakter 2.5D HD
Langkah ini cepat (~2–3 menit di 12 vCPU Colab Pro) dan otomatis melewati sprite yang sudah ada.

In [ ]:
import time

t0 = time.time()
print('=== 1/3 Generating high-res signage textures... ===')
run_logged(['python3', f'{REPO_DIR}/production/generate_textures.py'])

print('\n=== 2/3 Building master Blender 4.2 3D scene (three_unique_cities.blend)... ===')
run_logged([
    BLENDER_BIN, '--background', '--python-exit-code', '1',
    '--python', f'{REPO_DIR}/production/build_blender_scene.py'
], use_blender_env=True)

print(f'\n=== 3/3 Rendering 2.5D Open Free-Rig character sprites ({START_FRAME}..{END_FRAME} @ scale={SPRITE_SCALE_F})... ===')
run_logged([
    'python3', f'{REPO_DIR}/production/render_characters_seq.py',
    '--start', str(START_FRAME), '--end', str(END_FRAME),
    '--scale', str(SPRITE_SCALE_F), '--workers', str(CPU_WORKERS),
    '--skip-existing'
])

print(f'\nTahap persiapan selesai dalam {(time.time()-t0)/60:.1f} menit!')

## 5. Uji Cepat 3 Keyframe Cinema Full HD di GPU L4 (~10 Detik!)
Cek langsung kualitas **shadow, lighting, texture, clearcoat highlight, & bloom** Full HD `1080×1920` di 3 kota (Whittier, Coober Pedy Dugout, Setenil) sebelum menjalankan render penuh 2.283 frame!

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

TEST_DIR = Path('/content/l4_cinema_test')
TEST_DIR.mkdir(exist_ok=True)

run_logged([
    BLENDER_BIN, '--background', f'{REPO_DIR}/production/three_unique_cities.blend',
    '--python-exit-code', '1',
    '--python', f'{REPO_DIR}/production/render_blender_chunk.py', '--',
    '--frames', '35,1395,1855',
    '--res', RESOLUTION,
    '--samples', str(SAMPLES),
    '--device', DEVICE,
    '--quality', QUALITY_MODE,
    '--out-dir', str(TEST_DIR),
], use_blender_env=True, filter_keywords=('[Cycles GPU]', '[Cinema Upgrade]', 'Rendering ', '[Blender ', 'DONE!', 'Error', 'Traceback'))

# Tampilkan 3 hasil uji berdampingan
imgs = [Image.open(TEST_DIR / f'frame_{f:04d}.png').resize((360, 640), Image.Resampling.LANCZOS) for f in (35, 1395, 1855)]
strip = Image.new('RGB', (360 * 3, 640), (15, 15, 20))
for i, im in enumerate(imgs):
    strip.paste(im, (i * 360, 0))
display(strip)

## 6. 🚀 Render Penuh 2.283 Frame (76,1 Detik) Full HD 9:16 di GPU L4 OptiX
- Menggunakan `--skip-existing`: aman jika terputus, jalankan ulang sel ini untuk melanjutkan otomatis dari frame terakhir!
- Progres, kecepatan detik/frame, dan estimasi waktu selesai (ETA) ditampilkan setiap 10 frame.

In [ ]:
cmd = [
    BLENDER_BIN, '--background', f'{REPO_DIR}/production/three_unique_cities.blend',
    '--python-exit-code', '1',
    '--python', f'{REPO_DIR}/production/render_blender_chunk.py', '--',
    '--start', str(START_FRAME),
    '--end', str(END_FRAME),
    '--res', RESOLUTION,
    '--samples', str(SAMPLES),
    '--device', DEVICE,
    '--quality', QUALITY_MODE,
    '--skip-existing',
    '--out-dir', FRAMES_DIR,
]
print('Memulai render penuh NVIDIA L4 OptiX Cinema...\n')
run_logged(
    cmd,
    use_blender_env=True,
    filter_keywords=('[Cycles GPU]', '[Cinema Upgrade]', 'Rendering ', '[Blender ', '[Resume]', 'DONE!', 'Error', 'Traceback', 'Exception'),
)

## 7. Assemble Video Final Full HD MP4 (`CRF 16`, Audio Mix 256k AAC) & Unduh

In [ ]:
import os
from pathlib import Path
from PIL import Image
from IPython.display import display
from google.colab import files

done_frames = sorted(Path(FRAMES_DIR).glob('frame_*.png'))
print(f'Total frame ditemukan di {FRAMES_DIR}: {len(done_frames)} / {END_FRAME - START_FRAME + 1}')

run_logged([
    'python3', f'{REPO_DIR}/production/assemble_preview.py',
    '--mode', 'full_video',
    '--frames-dir', FRAMES_DIR,
    '--out', OUT_MP4,
    '--crf', str(CRF_VIDEO),
])

size_mb = os.path.getsize(OUT_MP4) / (1024 * 1024)
print(f'\n=== VIDEO FINAL SIAP! ===')
print(f'File   : {OUT_MP4}')
print(f'Ukuran : {size_mb:.1f} MB')

cs_path = Path(OUT_MP4).with_suffix('.contact_sheet.png')
if cs_path.exists():
    display(Image.open(cs_path))

# Otomatis unduh ke komputer Anda (selain sudah tersimpan aman di Google Drive)
files.download(OUT_MP4)